# TFMN Pooling Dashboard (Panel)

**Run cells in order:**

1. **Cell 1** — Install dependencies
2. **Cell 2** — Mount Google Drive (the drive that hosts CDS dataset)
3. **Cell 3** — Imports
4. **Cell 4** — Config + load data _(edit `MASTER_PKL` if needed)_
5. **Cell 5** — Launch dashboard
6. **Best practice** — Change output view to full screen for better experience

In [1]:
!pip install git+https://github.com/MassimoStel/emoatlas --quiet
!python -m spacy download en_core_web_lg
!pip install --upgrade pandas panel --quiet


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 248.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 32.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)




Mounted at /content/drive


In [3]:
import nltk
nltk.download('wordnet')

import ast, warnings, io
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — required for Panel
import matplotlib.pyplot as plt
import panel as pn
from emoatlas import EmoScores

warnings.filterwarnings('ignore')
pn.extension('tabulator', sizing_mode='stretch_width')
print('All imports OK')


[nltk_data] Downloading package wordnet to /root/nltk_data...


All imports OK


In [4]:
# ── CONFIG — edit MASTER_PARQUET ─────────────────────────────────────────────
import ast
import json
import numpy as np

MASTER_PARQUET = '/content/drive/MyDrive/Publications/LLMs_Persona/DashBoard-Colab/mergedEdgelists-withModelNames.parquet'

SOCIO_COLS = [
    'Model', 'mode', 'age',
    'gender', 'sexual_orientation', 'occupation',
    'city_of_living', 'employment_status', 'education_level',
    'parents_education', 'marital_status', 'children',
    'migration_status', 'psychological_feature',
    'religious_beliefs', 'time_onsocialmedia',
    'hobbies', 'ocean',
    'openness', 'conscientiousness', 'extraversion',
    'agreeableness', 'neuroticism',
]

TOPIC_OPTIONS = [
    'How to manage and dealing with fake content on social media.',
    'What is the gender gap in science? Should we address it? And how?',
    'What is stereotype threat in STEM? Should we address it? And how?',
    'How to manage and dealing with discussions about healthcare and COVID vaccines.',
]

LAYER_COL = {
    'opinion':           'opinion_Edge',
    'reasoning_summary': 'reasoning_summary_Edge',
}

ACCENT = '#2E86C1'

def parse_edge_list(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)): return []
    if isinstance(cell, list): return cell
    if isinstance(cell, np.ndarray): return cell.tolist()
    if isinstance(cell, str):
        cell = cell.strip()
        try: return json.loads(cell)
        except:
            try: return ast.literal_eval(cell)
            except: return []
    return []

def row_to_edges(row, col):
    edges = []
    for e in parse_edge_list(row[col]):
        if isinstance(e, str) and ',' in e:
            parts = [p.strip().lower() for p in e.split(',') if p.strip()]
            if len(parts) >= 2:
                edges.append((parts[0], parts[1]))
    return edges

master_df = pd.read_parquet(MASTER_PARQUET)
print(f'Loaded {len(master_df)} rows | Columns: {list(master_df.columns)}')

Loaded 226364 rows | Columns: ['Model', 'json_file', 'opinion_Edge', 'reasoning_summary_Edge', 'topic', 'mode', 'age', 'gender', 'sexual_orientation', 'occupation', 'city_of_living', 'employment_status', 'education_level', 'parents_education', 'marital_status', 'children', 'migration_status', 'psychological_feature', 'religious_beliefs', 'time_onsocialmedia', 'hobbies', 'ocean', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']


In [5]:
# ── Pooling Panel with MASTER_Parquet ─────────────────────────────────────────────
import ast
import json
import numpy as np
import io
pn.extension('tabulator', comms='colab', sizing_mode='stretch_width')

ACCENT = '#2E86C1'

# ── Parquet fix: normalize all string/category columns ───────────────────
str_cols = master_df.select_dtypes(include='object').columns
master_df[str_cols] = master_df[str_cols].apply(lambda c: c.str.strip())
cat_cols = master_df.select_dtypes(include='category').columns
master_df[cat_cols] = master_df[cat_cols].astype(str).apply(lambda c: c.str.strip())

# ── Sidebar widgets ───────────────────────────────────────────────────────
w_topic = pn.widgets.Select(
    name='Topic', options=TOPIC_OPTIONS,
    sizing_mode='stretch_width'
)
w_layer = pn.widgets.RadioButtonGroup(
    name='Layer', options=list(LAYER_COL.keys()),
    button_type='primary', sizing_mode='stretch_width'
)

socio_widgets = {}
for col in SOCIO_COLS:
    if col in master_df.columns:
        vals = sorted(master_df[col].dropna().astype(str).unique().tolist())
        socio_widgets[col] = pn.widgets.Select(
            name=col.replace('_', ' ').title(),
            options=['Any'] + vals,
            sizing_mode='stretch_width'
        )

btn_search = pn.widgets.Button(
    name='🔍  Search Personas', button_type='primary',
    sizing_mode='stretch_width', height=42
)

# ── Main area widgets ─────────────────────────────────────────────────────
w_persona = pn.widgets.Select(
    name='Select Persona', options={},
    sizing_mode='stretch_width'
)
w_plots = pn.widgets.CheckBoxGroup(
    name='Plots to Draw',
    options=['Emotional Flower', 'Forma Mentis Network', 'Ego Network', 'Mindset Stream'],
    value=['Emotional Flower', 'Forma Mentis Network', 'Ego Network', 'Mindset Stream'],
    inline=True
)
w_ego = pn.widgets.TextInput(
    name='Ego Keyword', value='vaccine', sizing_mode='stretch_width'
)
w_kw1 = pn.widgets.TextInput(
    name='Mindset Stream — Start', value='vaccine', sizing_mode='stretch_width'
)
w_kw2 = pn.widgets.TextInput(
    name='Mindset Stream — End', value='health', sizing_mode='stretch_width'
)
btn_draw = pn.widgets.Button(
    name='▶  Draw Figures', button_type='success',
    sizing_mode='stretch_width', height=42, disabled=True
)
btn_clear = pn.widgets.Button(
    name='✕  Clear', button_type='warning',
    sizing_mode='stretch_width', height=42
)

# ── Output panes ──────────────────────────────────────────────────────────
status_pane      = pn.pane.Markdown('_Run **Search Personas** to begin._',
                                     sizing_mode='stretch_width')
# persona_tbl_pane = pn.pane.DataFrame(pd.DataFrame(),
#                                       sizing_mode='stretch_width', max_height=280)


persona_tbl_pane = pn.widgets.Tabulator(
    pd.DataFrame(),
    sizing_mode='stretch_width',
    max_height=300,
    theme='site',
    show_index=False,
    pagination='local',
    page_size=10,
)


figures_col      = pn.Column(sizing_mode='stretch_width')
matched_cache    = {}


# ── Helpers ───────────────────────────────────────────────────────────────
def get_matching_rows():
    df = master_df.copy()
    # Parquet fix: normalize topic column before comparison
    df['topic'] = df['topic'].astype(str).str.strip()
    topic_val   = w_topic.value.strip()
    df = df[df['topic'] == topic_val]
    for col, ww in socio_widgets.items():
        if ww.value != 'Any':
            df = df[df[col].astype(str).str.strip() == ww.value.strip()]
    return df.reset_index(drop=True)


def row_to_edges(row, col):
    """Parse list of 'node1,node2' strings into (node1, node2) tuples."""
    edges = []
    for e in parse_edge_list(row[col]):
        if isinstance(e, str) and ',' in e:
            parts = [p.strip().lower() for p in e.split(',') if p.strip()]
            if len(parts) >= 2:
                edges.append((parts[0], parts[1]))
    return edges


def fig_to_panel(title):
    """Capture current matplotlib figure into a Panel PNG card."""
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close('all')
    buf.seek(0)
    return pn.Column(
        pn.pane.Markdown(f'### {title}'),
        pn.pane.PNG(buf, sizing_mode='stretch_width'),
        styles={
            'background': '#f4f8fb',
            'border': '1px solid #dde4ec',
            'border-radius': '10px',
            'padding': '14px',
            'margin-bottom': '18px',
        }
    )


# ── Search callback ───────────────────────────────────────────────────────
def on_search(event):
    figures_col.clear()
    matched = get_matching_rows()

    if matched.empty:
        status_pane.object      = '⚠️ **No personas match the selected filters.**'
        persona_tbl_pane.object = pd.DataFrame()
        w_persona.options       = {}
        btn_draw.disabled       = True
        return

    status_pane.object = f'✅ **Found {len(matched)} matching personas.**'

    display_cols = [c for c in SOCIO_COLS if c in matched.columns] + ['json_file']
    # persona_tbl_pane.object = (
    #     matched[display_cols].reset_index().rename(columns={'index': '#'})
    # )

    persona_tbl_pane.value = matched[display_cols].reset_index().rename(columns={'index': '#'})


    options = {}
    for i, (_, row) in enumerate(matched.iterrows()):
        label = f'#{i+1}  ' + '  |  '.join(
            f'{c}: {row[c]}' for c in SOCIO_COLS
            if c in row and pd.notna(row[c]) and str(row[c]).strip() not in ('nan', '')
        )
        options[label] = i
    w_persona.options = options
    w_persona.value   = list(options.values())[0] if options else None

    matched_cache['df']  = matched
    matched_cache['col'] = LAYER_COL[w_layer.value]
    btn_draw.disabled    = False


# ── Draw callback ─────────────────────────────────────────────────────────
def on_draw(event):
    figures_col.clear()
    status_pane.object = '⏳ **Rendering figures...**'

    if 'df' not in matched_cache:
        status_pane.object = '⚠️ **Please run Search Personas first.**'
        return

    matched     = matched_cache['df']
    col         = matched_cache['col']
    persona_idx = w_persona.value
    selected    = w_plots.value
    ego_kw      = w_ego.value.strip().lower()
    kw1         = w_kw1.value.strip().lower()
    kw2         = w_kw2.value.strip().lower()

    if persona_idx is None:
        status_pane.object = '⚠️ **No persona selected.**'
        return
    if not selected:
        status_pane.object = '⚠️ **No plots selected.**'
        return

    row   = matched.iloc[persona_idx]
    edges = row_to_edges(row, col)

    if not edges:
        status_pane.object = '⚠️ **No edges found for this persona.**'
        return

    persona_label = '  |  '.join(
        f'{c}: {row[c]}' for c in SOCIO_COLS
        if c in row and pd.notna(row[c]) and str(row[c]).strip() not in ('nan', '')
    )

    figures_col.append(
        pn.pane.Markdown(
            f'## Persona #{persona_idx+1}\n\n_{persona_label}_',
            sizing_mode='stretch_width',
            styles={
                'background': '#eaf4fb', 'border-radius': '10px',
                'padding': '14px', 'margin-bottom': '10px',
                'border': f'1px solid {ACCENT}'
            }
        )
    )

    # ── Build fmnt with correct edges AND vertices ────────────────────────
    emos      = EmoScores()
    seed_text = ' '.join([f'{a} {b}' for a, b in edges[:50]])
    fmnt      = emos.formamentis_network(seed_text)

    all_nodes = list(dict.fromkeys(
        node for edge in edges for node in edge
    ))
    fmnt = fmnt._replace(edges=edges, vertices=all_nodes)

    # 1. Emotional Flower
    if 'Emotional Flower' in selected:
        try:
            emos.draw_formamentis_flower(
                text=seed_text,
                title=f'Emotional Flower — Persona #{persona_idx+1}'
            )
            fig = plt.gcf()
            fig.set_size_inches(10, 10)
            figures_col.append(fig_to_panel('🌸 Emotional Flower'))
        except Exception as e:
            figures_col.append(pn.pane.Markdown(f'**🌸 Flower error:** `{e}`'))

    # 2. Forma Mentis Network
    if 'Forma Mentis Network' in selected:
        try:
            emos.draw_formamentis(
                fmn=fmnt,
                alpha_syntactic=0.5,
                alpha_hypernyms=0.3,
                alpha_synonyms=0.0,
                thickness=1
            )
            fig = plt.gcf()
            fig.set_size_inches(22, 22)
            for ax in fig.axes:
                for t in ax.texts:
                    t.set_fontsize(11)
            plt.suptitle(f'Forma Mentis Network — Persona #{persona_idx+1}', fontsize=13)
            figures_col.append(fig_to_panel('🕸️ Forma Mentis Network'))
        except Exception as e:
            figures_col.append(pn.pane.Markdown(f'**🕸️ TFMN error:** `{e}`'))

    # 3. Ego Network
    if 'Ego Network' in selected:
        try:
            fmnt_ego = emos.extract_word_from_formamentis(fmnt, ego_kw)
            emos.draw_formamentis(
                fmn=fmnt_ego,
                highlight=ego_kw,
                alpha_syntactic=0.4,
                alpha_hypernyms=0,
                alpha_synonyms=0,
                thickness=2,
            )
            fig = plt.gcf()
            fig.set_size_inches(16, 16)
            for ax in fig.axes:
                for t in ax.texts:
                    t.set_fontsize(12)
            plt.suptitle(f'Ego Network: "{ego_kw}" — Persona #{persona_idx+1}', fontsize=13)
            figures_col.append(fig_to_panel(f'🔍 Ego Network: "{ego_kw}"'))
        except Exception as e:
            figures_col.append(pn.pane.Markdown(f'**🔍 Ego error:** `{e}`'))

    # 4. Mindset Stream
    if 'Mindset Stream' in selected:
        try:
            emos.plot_mindset_stream(fmnt, start_node=kw1, end_node=kw2)
            fig = plt.gcf()
            fig.set_size_inches(14, 8)
            for ax in fig.axes:
                for t in ax.texts:
                    t.set_fontsize(11)
            plt.suptitle(
                f'Mindset Stream: "{kw1}" to "{kw2}" — Persona #{persona_idx+1}',
                fontsize=13
            )
            figures_col.append(fig_to_panel(f'🔗 Mindset Stream: "{kw1}" to "{kw2}"'))
        except Exception as e:
            figures_col.append(pn.pane.Markdown(f'**🔗 Stream error:** `{e}`'))

    status_pane.object = f'✅ **Done — {len(selected)} plot(s) rendered for Persona #{persona_idx+1}.**'


def on_clear(event):
    figures_col.clear()
    persona_tbl_pane.object = pd.DataFrame()
    status_pane.object      = '_Run **Search Personas** to begin._'
    w_persona.options       = {}
    btn_draw.disabled       = True


btn_search.on_click(on_search)
btn_draw.on_click(on_draw)
btn_clear.on_click(on_clear)


# ── Layout ────────────────────────────────────────────────────────────────
layout = pn.Row(
    pn.Column(
        pn.pane.Markdown('## 🔎 Filters'),
        w_topic,
        pn.pane.Markdown('**Layer**'),
        w_layer,
        pn.layout.Divider(),
        pn.pane.Markdown('**Sociodemographic Filters**\n\n_"Any" = no filter_'),
        *list(socio_widgets.values()),
        pn.layout.Divider(),
        btn_search,
        width=350,
        styles={
            'background': '#f0f4f8',
            'padding': '15px',
            'border-radius': '10px',
            'margin-right': '15px',
        }
    ),
    pn.Column(
        pn.pane.Markdown('## 👥 Matching Personas'),
        status_pane,
        persona_tbl_pane,
        pn.layout.Divider(),
        pn.pane.Markdown('## ⚙️ Visualization Settings'),
        w_persona,
        pn.pane.Markdown('**Select plots to draw**'),
        w_plots,
        pn.Row(w_ego, w_kw1, w_kw2, sizing_mode='stretch_width'),
        pn.Row(btn_draw, btn_clear, sizing_mode='stretch_width'),
        pn.layout.Divider(),
        pn.pane.Markdown('## 📊 Figures'),
        figures_col,
        sizing_mode='stretch_width',
        styles={'padding': '15px'},
    ),
    sizing_mode='stretch_width'
)

layout


Output hidden; open in https://colab.research.google.com to view.